# 05-03 RunnableLambda와 @chain 데코레이터: 함수를 체인으로 만들기

LCEL 파이프라인은 프롬프트, 모델, 파서 같은 내장 Runnable을 연결하는 데 강력하지만, **사용자 정의 로직**을 파이프라인에 끼워 넣고 싶을 때가 있어요. 예를 들어 입력 데이터를 전처리하거나, 중간 결과를 가공하거나, 외부 API를 호출하는 경우예요.

이때 `RunnableLambda`와 `@chain` 데코레이터를 사용하면 일반 파이썬 함수를 LCEL 체인의 구성 요소로 변환할 수 있어요.

## 학습 목표

- `RunnableLambda`로 파이썬 함수를 LCEL 파이프라인에서 사용할 수 있는 `Runnable` 객체로 변환하는 방법을 구현할 수 있어요
- 단일 인자 제약사항을 이해하고, 딕셔너리 래퍼 패턴으로 여러 인자를 처리하는 방법을 설명할 수 있어요
- `RunnableConfig`를 활용하여 태그, 메타데이터, 콜백 등 실행 설정을 체인 내부로 전파하는 원리를 이해할 수 있어요
- `@chain` 데코레이터로 함수를 더 간결하게 `Runnable`로 변환하고, `RunnableLambda`와의 차이를 비교할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- Ch01 LCEL 기초: 파이프 연산자(`|`)로 체인 구성하는 방법
- Ch05-01 `RunnablePassthrough`와 `RunnableParallel`의 역할
- 파이썬 데코레이터(Decorator) 문법 (`@`)
- 파이썬 딕셔너리(Dictionary) 패킹/언패킹

---

> **이 노트북의 흐름** — 아래 다이어그램은 일반 파이썬 함수가 어떻게 LCEL 체인의 구성 요소가 되는지 보여줘요.

```mermaid
flowchart LR
    F["일반 파이썬 함수<br/>def my_func(x):"]:::input
    RL["RunnableLambda(my_func)<br/>또는 @chain"]:::process
    R["Runnable 객체<br/>invoke/stream/batch 지원"]:::output
    C["LCEL 파이프라인<br/>prompt | model | my_func"]:::external

    F -->|"래핑"| RL
    RL -->|"변환 완료"| R
    R -->|"파이프 연결"| C

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, chain
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")


## RunnableLambda란?

LCEL 파이프라인의 모든 구성 요소는 `Runnable` 인터페이스를 따라야 해요. `ChatOpenAI`, `ChatPromptTemplate`, `StrOutputParser` 등은 이미 `Runnable`이지만, 사용자가 작성한 일반 파이썬 함수는 그렇지 않아요.

**`RunnableLambda`**는 일반 파이썬 함수를 `Runnable` 객체로 변환(래핑)해주는 클래스예요. 래핑된 함수는 `invoke()`, `stream()`, `batch()` 같은 Runnable 표준 메서드를 사용할 수 있고, 파이프 연산자(`|`)로 다른 Runnable과 연결할 수 있어요.

```python
# 래핑 전: 일반 파이썬 함수 (LCEL 파이프라인에 직접 사용 불가)
def my_func(text: str) -> int:
    return len(text)

# 래핑 후: Runnable 객체 (LCEL 파이프라인에서 사용 가능)
runnable_func = RunnableLambda(my_func)
runnable_func.invoke("hello")  # → 5
```

> 🔑 **핵심 개념**: `RunnableLambda`는 "일반 함수 → Runnable 객체" 변환기예요. LCEL 파이프라인은 Runnable만 연결할 수 있기 때문에, 사용자 정의 로직을 끼워 넣으려면 이 변환이 필요해요.

```mermaid
flowchart LR
    subgraph before["래핑 전"]
        PF["def length_function(text):<br/>    return len(text)"]:::input
    end
    
    subgraph after["래핑 후"]
        RL["RunnableLambda(length_function)"]:::process
        M1["invoke('hello') → 5"]:::output
        M2["stream('hello') → 5"]:::output
        M3["batch(['a','bb']) → [1,2]"]:::output
    end
    
    PF -->|"RunnableLambda()"| RL
    RL --> M1
    RL --> M2
    RL --> M3

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

## 1. RunnableLambda 기본 사용법

`RunnableLambda`는 사용자 정의 함수를 `Runnable`로 래핑하여 LCEL 파이프라인에서 사용할 수 있게 해줘요.

### 단일 인자 제약사항

`RunnableLambda`에 전달하는 함수는 **단일 인자만 받을 수 있어요**. 이는 LCEL 파이프라인의 설계 원칙인 "단일 입력 → 단일 출력"을 따르기 때문이에요.

여러 값이 필요한 경우에는 **딕셔너리 래퍼 패턴**을 사용해요:

| 패턴 | 함수 시그니처 | RunnableLambda 사용 |
|------|-------------|---------------------|
| 단일 인자 | `def fn(text: str)` | `RunnableLambda(fn)` — 직접 전달 가능 |
| 다중 인자 | `def fn(a: str, b: str)` | 직접 전달 불가 — 래퍼 필요 |
| 딕셔너리 래퍼 | `def fn(d: dict)` | `RunnableLambda(fn)` — 딕셔너리로 값 전달 |

> ⚠️ **주의**: `RunnableLambda(fn)`에 전달하는 함수가 2개 이상의 인자를 받으면 런타임 오류가 발생해요. 딕셔너리 하나를 받고 내부에서 값을 꺼내는 패턴으로 우회해요.

> 🔑 **핵심 개념**: LCEL 파이프라인의 모든 단계는 "단일 입력 → 단일 출력" 원칙을 따라요. 여러 값이 필요하면 딕셔너리로 묶어서 전달하는 것이 표준 패턴이에요.

아래 코드에서 `length_function`(단일 인자)과 `multiple_length_function`(딕셔너리 래퍼)의 차이를 비교해 보세요.

In [3]:
# ============================================================
# TODO: RunnableLambda로 사용자 정의 함수를 체인에 연결하세요
# 힌트: RunnableLambda(fn) — fn은 반드시 단일 인자를 받아야 함
#       여러 인자가 필요하면 딕셔너리로 묶고 래퍼 함수를 사용
# 예상 결과: "3 + 9 equals 12."
# ============================================================

# ---------------------------------------------------
# RunnableLambda: 사용자 정의 함수를 Runnable로 래핑하기
# ---------------------------------------------------

# 1단계: 단일 인자를 받는 함수 정의
# RunnableLambda에 직접 전달 가능
def length_function(text: str) -> int:
    """텍스트의 길이를 반환하는 함수"""
    return len(text)


# 2단계: 두 인자를 받는 함수 (직접 RunnableLambda에 전달 불가)
def _multiple_length_function(text1: str, text2: str) -> int:
    """두 텍스트의 길이를 곱하는 함수 (2개 인자 — 직접 사용 불가)"""
    return len(text1) * len(text2)


# 3단계: 딕셔너리를 받는 래퍼 함수 (RunnableLambda에 전달 가능)
# 핵심: 딕셔너리 하나를 받고 내부에서 여러 값을 꺼내는 패턴
def multiple_length_function(_dict: dict) -> int:
    """딕셔너리에서 "text1"과 "text2"를 추출하여 길이를 곱하는 래퍼 함수"""
    return _multiple_length_function(_dict["text1"], _dict["text2"])


# 4단계: 프롬프트 템플릿 생성
prompt = ChatPromptTemplate.from_template("what is {a} + {b}?")

# 5단계: 체인 구성
# - itemgetter: 딕셔너리에서 특정 키 추출 (Runnable로 자동 변환)
# - RunnableLambda: 사용자 정의 함수를 파이프라인에 연결
lambda_chain = (
    {
        "a": itemgetter("input_1") | RunnableLambda(length_function),
        "b": {
            "text1": itemgetter("input_1"),
            "text2": itemgetter("input_2")
        } | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
    | StrOutputParser()
)

print("=" * 60)
print("✅ 체인 구성 완료")
print("=" * 60)


✅ 체인 구성 완료


In [4]:
# 체인 실행
result = lambda_chain.invoke({"input_1": "bar", "input_2": "gah"})

print("=" * 60)
print("📥 실행 결과")
print("=" * 60)
print(f"입력: {{'input_1': 'bar', 'input_2': 'gah'}}")
print(f"출력: {result}")
print()
print("💡 설명:")
print("  - 'bar'의 길이: 3")
print("  - 'gah'의 길이: 3")
print("  - 3 + (3 * 3) = 3 + 9 = 12")


📥 실행 결과
입력: {'input_1': 'bar', 'input_2': 'gah'}
출력: 3 + 9 equals 12.

💡 설명:
  - 'bar'의 길이: 3
  - 'gah'의 길이: 3
  - 3 + (3 * 3) = 3 + 9 = 12


## 2. 실행 설정(RunnableConfig) 활용

### RunnableConfig란?

`RunnableConfig`는 LangChain에서 **체인의 실행 환경을 정의하는 설정 객체**예요. 체인을 실행할 때 태그(Tag), 메타데이터(Metadata), 콜백(Callback) 등을 함께 전달하면, 체인 내부의 모든 Runnable이 이 설정을 공유해요.

쉽게 비유하면, `RunnableConfig`는 택배 송장과 비슷해요. 택배(입력 데이터)와 함께 송장(config)을 붙이면, 물류 허브(체인의 각 단계)를 거칠 때마다 송장 정보가 함께 전달돼요.

### RunnableConfig의 주요 필드

| 필드 | 타입 | 설명 |
|------|------|------|
| `tags` | `list[str]` | 실행 추적용 태그. LangSmith에서 특정 실행을 필터링할 때 사용해요 |
| `metadata` | `dict` | 사용자 정의 메타데이터. 요청 ID, 사용자 정보 등을 기록할 수 있어요 |
| `callbacks` | `list[BaseCallbackHandler]` | 실행 이벤트(시작, 종료, 오류)를 모니터링하는 콜백 핸들러 |
| `run_name` | `str` | 실행 이름. LangSmith 대시보드에서 표시되는 이름이에요 |
| `max_concurrency` | `int` | `batch()` 호출 시 최대 동시 실행 수 |
| `recursion_limit` | `int` | 재귀 호출 제한 (기본값: 25). 무한 루프 방지에 사용해요 |
| `configurable` | `dict` | `ConfigurableField`로 정의한 동적 설정값 (Ch05-06에서 다룰 예정) |

### config 전파(Propagation) 원리

`chain.invoke(input, config=config)`처럼 호출하면, `config`가 체인 내부의 **모든 Runnable에 자동 전달**돼요. 별도의 코드 없이도 프롬프트, 모델, 파서 각각이 같은 `config`를 받아요.

`RunnableLambda` 안의 사용자 정의 함수에서 이 `config`에 접근하려면, **함수의 두 번째 매개변수**로 `config`를 선언하면 돼요. `RunnableLambda`가 호출 시 자동으로 주입해줘요.

```python
# config를 받지 않는 함수 (기본)
def my_func(text: str) -> str:
    return text.upper()

# config를 받는 함수 (두 번째 인자로 선언)
def my_func_with_config(text: str, config: RunnableConfig) -> str:
    print(f"태그: {config.get('tags', [])}")
    return text.upper()
```

> 🔑 **핵심 개념**: `RunnableConfig`는 체인의 "실행 환경"이에요. 모든 Runnable이 공유하는 설정 정보를 담고 있어요.

> 🎯 **강의 포인트**: `config` 매개변수는 반드시 **두 번째 인자**로 받아야 해요. 첫 번째 인자는 항상 입력 데이터이고, `RunnableLambda`가 `config`를 자동으로 주입해줘요.

> 💡 **실무 팁**: LangSmith 추적 시 `tags`와 `metadata`를 활용하면 특정 요청을 필터링할 수 있어요. 예를 들어 `tags=["production", "user-123"]`으로 특정 사용자의 요청만 모아볼 수 있어요.

> ⚠️ **주의**: `config`를 `invoke()`에 명시적으로 전달하지 않아도 체인 내부에서는 기본 config가 자동 생성돼요. 하지만 태그나 메타데이터를 활용하려면 직접 전달해야 해요.

### config 전파 흐름

아래 다이어그램은 `config`가 체인의 각 단계로 어떻게 전달되는지 보여줘요.

```mermaid
flowchart TD
    U["사용자 호출<br/>chain.invoke(input, config={<br/>  tags: ['json-fix'],<br/>  metadata: {source: 'api'}<br/>})"]:::input

    subgraph chain["체인 내부 — config가 모든 단계에 전파"]
        S1["RunnableLambda<br/>parse_or_fix(text, config)<br/>config.tags → ['json-fix']"]:::process
        S2["ChatPromptTemplate<br/>config 자동 수신"]:::process
        S3["ChatOpenAI<br/>config.tags로 LangSmith 추적"]:::external
        S4["StrOutputParser<br/>config 자동 수신"]:::process
    end

    R["결과 반환"]:::output

    U -->|"config 전달"| S1
    S1 -->|"내부 체인 invoke(input, config)"| S2
    S2 --> S3
    S3 --> S4
    S4 --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

### 실습: JSON 파싱 + 자동 수정 with RunnableConfig

아래 코드는 `RunnableConfig`의 실전 활용 예시예요. 잘못된 JSON을 파싱하다 실패하면 LLM에게 수정을 요청하는데, 이때 `config`를 내부 체인에 그대로 전달하여 태그와 메타데이터가 유지되도록 해요.

In [5]:
# ============================================================
# TODO: RunnableConfig를 받는 함수를 정의하고 RunnableLambda로 래핑하세요
# 힌트: def fn(text: str, config: RunnableConfig): ...  → 두 번째 인자로 config 수신
# 예상 결과: 잘못된 JSON을 LLM이 수정 시도 후 결과 출력
# ============================================================

from langchain_core.runnables import RunnableConfig
import json

# ---------------------------------------------------
# RunnableConfig 실전 활용
# 시나리오: 잘못된 JSON을 파싱하고, 실패 시 LLM에게 수정 요청
#           이때 config(태그, 메타데이터)를 내부 체인에 전파
# ---------------------------------------------------

# 1단계: config를 두 번째 인자로 받는 함수 정의
# - 첫 번째 인자: 입력 데이터 (RunnableLambda의 단일 인자 규칙)
# - 두 번째 인자: RunnableConfig (RunnableLambda가 자동 주입)
def parse_or_fix(text: str, config: RunnableConfig):
    """JSON 텍스트를 파싱하고, 실패하면 LLM으로 수정 시도"""
    # 2단계: 내부에서 사용할 수정 체인 생성
    fixing_chain = (
        ChatPromptTemplate.from_template(
            """다음 텍스트를 올바른 JSON 형식으로 수정해주세요:
            텍스트: 
                {input}
            오류: 
                {error}
            설명 없이 수정된 JSON만 반환해주세요.
            
            출력 예시:
                {{"a": "abc", "b": 2}}"""
        )
        | ChatOpenAI(model="gpt-4o-mini")
        | StrOutputParser()
    )
    
    # 3단계: 최대 3번 시도 (파싱 → 실패 시 LLM 수정 → 재파싱)
    for attempt in range(3):
        try:
            print(text)
            # JSON 형식으로 파싱 시도
            return json.loads(text)
        except Exception as e:
            # 4단계: 파싱 실패 시 LLM으로 수정 요청
            # config를 그대로 전달 → 태그, 메타데이터가 내부 체인에도 전파
            text = fixing_chain.invoke({"input": text, "error": str(e)}, config)
            print(f"시도 {attempt + 1}: config 전달됨")
    
    # 모든 시도 실패
    return {"error": "Failed to parse"}


# 5단계: RunnableLambda로 래핑하여 실행
# config 딕셔너리를 통해 태그와 메타데이터 전달
# → parse_or_fix 함수의 두 번째 인자로 자동 주입됨
# → 내부 fixing_chain.invoke() 호출 시에도 같은 config가 전파됨
output = RunnableLambda(parse_or_fix).invoke(
    input="{foo:: bar}",  # 잘못된 JSON 형식 (콜론이 2개)
    config={
        "tags": ["json-parsing", "auto-fix"],
        "metadata": {"source": "user-input"}
    }
)

print("=" * 60)
print("📥 JSON 파싱 결과")
print("=" * 60)
print(f"수정된 결과: {output}")

{foo:: bar}
시도 1: config 전달됨
{"foo": "bar"}
📥 JSON 파싱 결과
수정된 결과: {'foo': 'bar'}


## 3. @chain 데코레이터 — 더 간결한 문법

`RunnableLambda`로 함수를 래핑하는 방법을 배웠으니, 이제 같은 기능을 더 간결하게 표현하는 `@chain` 데코레이터를 살펴볼게요.

### @chain은 RunnableLambda의 축약 표현

`@chain` 데코레이터를 함수 위에 붙이면, 그 함수가 자동으로 `RunnableLambda`로 래핑돼요. 내부적으로는 동일한 동작을 하지만, 코드가 더 간결해요.

```python
# 방법 1: RunnableLambda로 명시적 래핑
def my_func(text: str) -> str:
    return text.upper()
runnable = RunnableLambda(my_func)

# 방법 2: @chain 데코레이터로 자동 래핑 (위와 동일)
@chain
def my_func(text: str) -> str:
    return text.upper()
# my_func은 이미 Runnable 객체!
```

### 언제 어떤 방법을 사용할까?

| 상황 | 추천 방법 | 이유 |
|------|----------|------|
| 함수 자체가 하나의 체인 역할 | `@chain` | 함수 정의와 Runnable 변환을 한 번에 처리 |
| 기존 함수를 체인 중간에 삽입 | `RunnableLambda` | 이미 정의된 함수를 파이프라인에 연결 |
| 람다(lambda) 함수 사용 | `RunnableLambda` | 데코레이터는 람다에 적용 불가 |
| 여러 체인을 순차 호출하는 함수 | `@chain` | 내부 로직이 복잡할 때 가독성이 좋음 |

> 🎯 **강의 포인트**: `@chain`과 `RunnableLambda`는 기능적으로 100% 동일해요. 선택 기준은 "함수를 체인으로 정의할 때"는 `@chain`, "체인 중간에 함수를 삽입할 때"는 `RunnableLambda`를 사용하는 것이 관용적이에요.

> 💡 **실무 팁**: `@chain` 데코레이터를 사용하면 LangSmith에서 함수 이름이 자동으로 실행 이름(`run_name`)으로 설정돼요. 디버깅할 때 어떤 함수에서 오류가 발생했는지 쉽게 파악할 수 있어요.

In [6]:
# ============================================================
# TODO: @chain 데코레이터를 사용하여 두 체인을 순차 실행하는 함수를 작성하세요
# 힌트: @chain 데코레이터를 함수 위에 붙이면 자동으로 Runnable로 변환됨
# 예상 결과: "양자역학" 입력 → 설명 생성 → 이모지 인스타그램 게시글 변환
# ============================================================

# ---------------------------------------------------
# @chain 데코레이터: 함수를 Runnable로 자동 변환
# 시나리오: 주제 → 설명 생성 → 인스타그램 게시글로 변환 (2단계 순차 처리)
# ---------------------------------------------------

# 1단계: 프롬프트 템플릿 정의
prompt1 = ChatPromptTemplate.from_template("{topic}에 대해 짧게 한글로 설명해주세요.")
prompt2 = ChatPromptTemplate.from_template(
    "{sentence}를 이모지를 활용한 인스타그램 게시글로 만들어주세요."
)


# 2단계: @chain 데코레이터로 함수를 Runnable로 자동 변환
# - @chain을 붙이면 invoke(), stream(), batch() 메서드가 자동 부여돼요
# - 내부에서 다른 체인을 invoke()로 호출하여 순차 처리하는 패턴이에요
# - @chain == RunnableLambda(custom_chain) 래핑과 동일하지만 더 간결해요
@chain
def custom_chain(text: str) -> str:
    """주제 설명을 생성하고 인스타그램 게시글로 변환하는 체인"""
    # 첫 번째 체인: 주제 설명 생성
    chain1 = prompt1 | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()
    output1 = chain1.invoke({"topic": text})
    
    # 두 번째 체인: 인스타그램 게시글 생성
    chain2 = prompt2 | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()
    output2 = chain2.invoke({"sentence": output1})
    
    return output2


print("=" * 60)
print("✅ @chain 데코레이터로 체인 생성 완료")
print("=" * 60)
print("custom_chain은 이제 Runnable 객체입니다!")
print()

✅ @chain 데코레이터로 체인 생성 완료
custom_chain은 이제 Runnable 객체입니다!



In [7]:
# @chain으로 만든 체인 실행
result = custom_chain.invoke("양자역학")

print("=" * 60)
print("📥 실행 결과")
print("=" * 60)
print(result)
print()
print("💡 @chain 데코레이터를 사용하면:")
print("  - 함수가 자동으로 Runnable로 변환됨")
print("  - invoke(), stream() 등 Runnable 메서드 사용 가능")
print("  - LangSmith 추적에서 중첩된 체인 호출 확인 가능")


📥 실행 결과
🌌 양자역학의 세계에 오신 것을 환영합니다! ✨ 

양자역학은 미시 세계, 즉 원자와 아원자 입자의 신비로운 운동과 상호작용을 탐구하는 분야예요. 🔬💫 

🔍 전통적인 고전역학과는 달리, 양자역학에서는 입자가 특정한 위치와 속도를 동시에 가질 수 없답니다! 그 대신, 우리는 확률로 모든 걸 설명하죠. 🎲📊 

이러한 개념은 불확정성 원리(🌀)와 파동-입자 이중성(🌊⚛️)으로 잘 알려져 있어요. 

양자역학은 원자, 분자, 고체 물리학 등 다양한 분야에서 중요한 역할을 하며, 현대 기술인 반도체(💻), 레이저(🔦), MRI(🩻) 등에도 깊게 사용되고 있습니다!

미시 세계의 매혹적인 비밀을 함께 알아보아요! 🤩💖 #양자역학 #과학 #미시세계 #물리학 #기술혁신

💡 @chain 데코레이터를 사용하면:
  - 함수가 자동으로 Runnable로 변환됨
  - invoke(), stream() 등 Runnable 메서드 사용 가능
  - LangSmith 추적에서 중첩된 체인 호출 확인 가능


## 정리

### 이 노트북에서 다룬 핵심 개념

| 개념 | 핵심 메서드/문법 | 용도 |
|------|-----------------|------|
| `RunnableLambda` | `RunnableLambda(fn)` | 일반 함수를 Runnable로 래핑하여 LCEL 파이프라인에 삽입 |
| `@chain` 데코레이터 | `@chain` + 함수 정의 | 함수를 Runnable로 자동 변환 (RunnableLambda의 축약 표현) |
| `RunnableConfig` | `config={"tags": [...], "metadata": {...}}` | 실행 설정(태그, 메타데이터, 콜백)을 체인 전체에 전파 |
| 딕셔너리 래퍼 패턴 | `def fn(d: dict)` | 단일 인자 제약을 우회하여 여러 값을 전달 |

### RunnableLambda vs @chain 데코레이터 비교

| 항목 | `RunnableLambda` | `@chain` 데코레이터 |
|------|-----------------|---------------------|
| 사용법 | `RunnableLambda(fn)` | `@chain` 데코레이터 추가 |
| 코드 간결성 | 명시적 (래핑 코드 필요) | 간결 (데코레이터 한 줄) |
| 람다 함수 지원 | 가능 | 불가 |
| RunnableConfig 전달 | 가능 (두 번째 인자) | 가능 (두 번째 인자) |
| 적합한 상황 | 체인 중간에 함수 삽입 | 함수 자체를 체인으로 정의 |
| LangSmith 실행 이름 | 함수명 자동 설정 | 함수명 자동 설정 |

### RunnableConfig 주요 필드 요약

| 필드 | 용도 |
|------|------|
| `tags` | LangSmith에서 실행 필터링 |
| `metadata` | 요청 ID, 사용자 정보 등 사용자 정의 데이터 |
| `callbacks` | 실행 이벤트 모니터링 |
| `run_name` | LangSmith 대시보드에 표시할 이름 |
| `max_concurrency` | batch() 시 최대 동시 실행 수 |
| `configurable` | ConfigurableField 동적 설정값 |

### 핵심 요점

- `RunnableLambda`는 함수를 파이프라인 중간에 명시적으로 삽입할 때 사용해요
- 함수는 반드시 **단일 인자**를 받아야 해요. 여러 값이 필요하면 딕셔너리로 묶어서 전달해요
- `RunnableConfig`는 체인의 실행 환경 설정이에요. `config`를 두 번째 인자로 선언하면 `RunnableLambda`가 자동으로 주입해요
- `@chain` 데코레이터는 `RunnableLambda`와 동일한 기능이지만 더 읽기 쉬운 코드를 만들어줘요
- 두 방법 모두 `invoke()`, `stream()`, `batch()` 등 `Runnable` 메서드를 그대로 사용할 수 있어요

### 다음 노트북 예고

다음 노트북(`04-Routing.ipynb`)에서는 입력 데이터의 특성에 따라 서로 다른 처리 경로를 선택하는 라우팅(Routing) 패턴을 배워요. `RunnableLambda`가 조건부 분기에 어떻게 활용되는지 살펴볼게요.